# Fase 2 — Preparação dos dados

**Objetivo da fase:** auditar o dataset fornecido, limpar, padronizar e dividir em treino, validação e teste.

## 2.1 Preparar o ambiente

### 2.1.1 Configurar o Google Colab

Abrir o link do notebook e ir em *Arquivo → Salvar uma cópia no Drive*. A partir daqui, trabalhar sempre na cópia.

Em *Ambiente de execução → Alterar o tipo de ambiente de execução*, selecionar **Versão 2026.10**, a mesma da Fase 1. Esta fase não usa GPU.

O secret `HF_TOKEN` criado na Fase 1 já fica disponível para todos os notebooks da conta. No painel lateral, ícone de chave, ligar *Acesso ao notebook* para o `HF_TOKEN` e rodar a célula abaixo para verificar a configuração.

In [ ]:
from google.colab import userdata
from huggingface_hub import login, whoami

login(token=userdata.get("HF_TOKEN"))

conta = whoami()
print(f"Conta:         {conta['name']}")
print(f"Tipo de token: {conta['auth']['accessToken']['role']}")
# Esperado: o nome da conta e "write"

### 2.1.2 Instalar as bibliotecas

Instalar as bibliotecas através das linhas de comando abaixo:

In [ ]:
# Versões fixadas para o runtime 2026.10 do Colab (ver README)
!pip install -q datasets==4.8.5 scikit-learn==1.6.1 huggingface_hub==1.33.0

## 2.2 Carregar o dataset

### 2.2.1 Formato do exemplo

O dataset fornecido é o histórico de triagens feitas manualmente pelos analistas L1. Cada linha é um alerta já enriquecido, com a triagem registrada pelo analista:

```json
{
  "template_id": "edr_powershell_codificado_office",
  "alerta": "EDR | Host: SRV-FIN-04 | Usuário: j.silva | Processo: powershell.exe -enc JAB... | Pai: outlook.exe",
  "contexto": "Ativo: servidor financeiro (criticidade: crítico) | Usuário: analista financeiro, sem privilégio admin | Reputação: desconhecida | Ocorrências em 7 dias: 1",
  "triagem": {"veredito": "verdadeiro_positivo", "severidade": "critica", "justificativa": "PowerShell codificado iniciado pelo Outlook é padrão típico de macro maliciosa, e ocorreu em servidor financeiro, ativo crítico."}
}
```

O campo `template_id` identifica o **padrão de alerta** (a regra de detecção e o comportamento observado). Exemplos do mesmo template mudam host, usuário e ativo, mas descrevem o mesmo sinal técnico. Esse campo é usado na auditoria (seção 2.3.4) e na divisão dos dados (seção 2.4).

### 2.2.2 Carregar do Hub

Carregar o arquivo do Hub e separar a triagem em colunas para facilitar a auditoria:

In [ ]:
import pandas as pd
from datasets import load_dataset

DATASET_ID = "poliana-moraes/alertas-soc-triagem"

dados = load_dataset(DATASET_ID, data_files="hf-dataset/alertas_soc.jsonl", split="train")
print(len(dados))  # 1560
print(dados[0])

bruto = dados.to_pandas()
df = pd.concat([bruto.drop(columns="triagem"), pd.DataFrame(bruto["triagem"].tolist())], axis=1)

print(f"Templates: {df['template_id'].nunique()}")
df.head()

## 2.3 Auditoria

A auditoria segue uma ordem: primeiro padronizar, porque grafias diferentes escondem duplicatas e categorias válidas. Depois remover o que não tem conserto, corrigir o que tem e, por fim, conferir se sobram exemplos suficientes de cada classe.

Cada etapa registra quantos exemplos foram afetados no dicionário `relatorio`, que no final vira o relatório de auditoria.

### 2.3.1 Padronizar textos e categorias

Analistas diferentes registram a mesma categoria de jeitos diferentes (`Crítica`, `CRITICA`, `critica `). Para o modelo, cada grafia seria uma classe nova. Padronizar:

- **Textos** (`alerta`, `contexto`, `justificativa`): espaços repetidos viram um só e as pontas são aparadas.
- **Categorias** (`veredito`, `severidade`): minúsculas, sem acento e com `_` no lugar de espaço ou hífen.

In [ ]:
import re
import unicodedata

relatorio = {"Exemplos no arquivo bruto": len(df)}


def padronizar_texto(s: str | None) -> str | None:
    return None if s is None else re.sub(r"\s+", " ", s).strip()


def padronizar_categoria(s: str | None) -> str | None:
    if s is None:
        return None
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"[\s-]+", "_", s.strip().lower())


antes = df.copy()
for col in ["alerta", "contexto", "justificativa"]:
    df[col] = df[col].map(padronizar_texto)
for col in ["veredito", "severidade"]:
    df[col] = df[col].map(padronizar_categoria)

alterados = (df.fillna("") != antes.fillna("")).any(axis=1)
relatorio["Padronizados (texto ou categoria)"] = int(alterados.sum())

print(f"Exemplos alterados: {alterados.sum()}")
print("Severidades antes:", sorted(antes["severidade"].dropna().unique()))
print("Severidades depois:", sorted(df["severidade"].dropna().unique()))

### 2.3.2 Triagem inválida

Depois da padronização, o que ainda está fora das categorias permitidas não tem conserto: não dá para saber o que o analista quis dizer com `urgente` ou `inconclusivo`. Triagem sem justificativa também é removida, porque o modelo precisa aprender a justificar.

As categorias são as mesmas da validação da Fase 1:

In [ ]:
VEREDITOS = {"falso_positivo", "verdadeiro_positivo"}
SEVERIDADES = {"baixa", "media", "alta", "critica"}

invalida = (
    ~df["veredito"].isin(VEREDITOS)
    | ~df["severidade"].isin(SEVERIDADES)
    | df["justificativa"].fillna("").eq("")
)
relatorio["Removidos: triagem inválida"] = int(invalida.sum())

print(f"Triagens inválidas: {invalida.sum()}")
print(df.loc[invalida, ["veredito", "severidade", "justificativa"]].head(10))

df = df[~invalida].reset_index(drop=True)

### 2.3.3 Duplicatas

Duplicata é o mesmo par `alerta` + `contexto` registrado mais de uma vez. Ela dá peso dobrado a um exemplo no treino e, se uma cópia cair no treino e outra no teste, o modelo acerta o teste por memória.

Comparar a contagem no arquivo bruto com a contagem depois da padronização mostra por que padronizar vem primeiro:

In [ ]:
CHAVE = ["alerta", "contexto"]

duplicatas_bruto = bruto.duplicated(subset=CHAVE).sum()
duplicada = df.duplicated(subset=CHAVE, keep="first")
relatorio["Removidos: duplicatas"] = int(duplicada.sum())

print(f"Duplicatas no arquivo bruto:      {duplicatas_bruto}")
print(f"Duplicatas depois de padronizar: {duplicada.sum()}")

df = df[~duplicada].reset_index(drop=True)

### 2.3.4 Rótulos inconsistentes com a política

Um rótulo pode estar no formato certo e mesmo assim errado: o analista aplicou a política de severidade da Fase 1 de forma diferente. Para encontrar esses casos, a severidade esperada de cada exemplo é recalculada pela política.

A política depende do **sinal técnico** e da **criticidade do ativo**. A criticidade está escrita no `contexto`. O sinal técnico não está, mas é o mesmo para todos os exemplos de um template. Então:

1. Para cada exemplo, descobrir qual sinal explica o rótulo registrado, invertendo a tabela da política. Ex.: ativo comum rotulado `alta` implica sinal forte.
2. Em cada template, o sinal mais votado é o sinal do template. Com 25 exemplos por template, alguns rótulos errados não mudam a maioria.
3. Recalcular veredito e severidade pela política e marcar os exemplos que divergem.

| Sinal técnico | Ativo crítico | Ativo comum | Ativo de laboratório |
|---|---|---|---|
| Forte | critica | alta | media |
| Moderado | alta | media | baixa |
| Fraco e recorrente | media | baixa | baixa (falso_positivo) |

Só o sinal fraco em ativo de laboratório tem veredito `falso_positivo`. Os demais são `verdadeiro_positivo`.

In [ ]:
POLITICA = {
    ("forte", "crítico"): "critica", ("forte", "comum"): "alta", ("forte", "laboratório"): "media",
    ("moderado", "crítico"): "alta", ("moderado", "comum"): "media", ("moderado", "laboratório"): "baixa",
    ("fraco", "crítico"): "media", ("fraco", "comum"): "baixa", ("fraco", "laboratório"): "baixa",
}


def triagem_esperada(sinal: str, criticidade: str) -> tuple[str, str]:
    veredito = "falso_positivo" if (sinal, criticidade) == ("fraco", "laboratório") else "verdadeiro_positivo"
    return veredito, POLITICA[(sinal, criticidade)]


# (criticidade, veredito, severidade) -> sinal que explica esse rótulo
SINAL_DO_ROTULO = {(c, *triagem_esperada(s, c)): s for s, c in POLITICA}

df["criticidade"] = df["contexto"].str.extract(r"criticidade: (\w+)", expand=False)
df["sinal_do_rotulo"] = [SINAL_DO_ROTULO.get(k) for k in zip(df["criticidade"], df["veredito"], df["severidade"])]

votos = df.groupby("template_id")["sinal_do_rotulo"]
sinal_template = votos.agg(lambda s: s.mode()[0])
concordancia = votos.agg(lambda s: (s == s.mode()[0]).mean())
print(f"Templates por sinal: {sinal_template.value_counts().to_dict()}")
print(f"Menor concordância com o sinal do template: {concordancia.min():.0%}")

df["sinal"] = df["template_id"].map(sinal_template)
esperado = [triagem_esperada(s, c) for s, c in zip(df["sinal"], df["criticidade"])]
df["veredito_esperado"] = [v for v, _ in esperado]
df["severidade_esperada"] = [s for _, s in esperado]

inconsistente = (df["veredito"] != df["veredito_esperado"]) | (df["severidade"] != df["severidade_esperada"])
print(f"\nRótulos inconsistentes com a política: {inconsistente.sum()}")
pd.crosstab(df.loc[inconsistente, "severidade_esperada"], df.loc[inconsistente, "severidade"],
            rownames=["esperada"], colnames=["registrada"])

A tabela mostra, para os rótulos inconsistentes, a severidade esperada (linhas) e a registrada pelo analista (colunas). A linha `critica` é a que mais importa: são ataques em ativos críticos que foram para o fim da fila.

Inspecionar alguns casos antes de corrigir:

In [ ]:
colunas = ["template_id", "criticidade", "severidade", "severidade_esperada", "veredito", "veredito_esperado"]
df.loc[inconsistente, colunas].head(10)

> **Ponto de discussão:** um rótulo errado no treino ensina o modelo a errar. Em segurança, um único padrão "crítico rotulado como baixa" repetido vira um ponto cego do modelo.

**Decisão: corrigir em vez de remover.** A política é determinística, então o rótulo certo é conhecido. Remover também tiraria do treino justamente os críticos, que já são a classe mais rara. A justificativa é mantida, porque ela descreve o sinal técnico e o contexto, que não mudam com a correção.

In [ ]:
relatorio["Corrigidos: rótulo inconsistente com a política"] = int(inconsistente.sum())
relatorio["Corrigidos: críticos rotulados com severidade menor"] = int(
    (inconsistente & (df["severidade_esperada"] == "critica")).sum()
)

df["veredito"] = df["veredito_esperado"]
df["severidade"] = df["severidade_esperada"]

### 2.3.5 Distribuição de classes

Conferir se há exemplos suficientes de `critica` e de `falso_positivo`, as duas classes que mais pesam nas metas da Fase 1:

In [ ]:
print(df["severidade"].value_counts().reindex(["critica", "alta", "media", "baixa"]))
print()
print(df["veredito"].value_counts())
print()
print(pd.crosstab(df["criticidade"], df["severidade"]).reindex(columns=["critica", "alta", "media", "baixa"]))

**Leitura:** `critica` (cerca de 11% dos exemplos) e `falso_positivo` (cerca de 8%) são as classes menores, mas cada uma tem mais de 100 exemplos e aparece em todos os templates de sinal forte e de sinal fraco, respectivamente. É suficiente para o modelo aprender o padrão. Se o recall de "crítica" ficar baixo na Fase 6, o tamanho dessa classe é a primeira hipótese a registrar como causa.

### 2.3.6 Relatório de auditoria

Resumo de tudo o que foi encontrado e feito:

In [ ]:
relatorio["Exemplos limpos"] = len(df)
pd.Series(relatorio, name="exemplos").to_frame()

## 2.4 Divisão treino/validação/teste

### 2.4.1 Dividir por template

Proporção: **70% / 15% / 15%**.

A divisão é feita por `template_id`: todos os exemplos de um template ficam no mesmo split. Isso evita *data leakage*: se variações do mesmo template caem no treino e no teste, o modelo "acerta" o teste por memorização e a métrica fica inflada. Com a divisão por template, o teste mede se o modelo aprendeu a **política**, aplicada a padrões de alerta que ele nunca viu.

Além disso, a divisão é feita separadamente para cada sinal técnico (14 / 3 / 3 dos 20 templates de cada sinal). Sem isso, o sorteio poderia deixar o teste sem nenhum template de sinal fraco e, portanto, sem nenhum `falso_positivo`.

In [ ]:
import numpy as np

SEED = 42
PROPORCAO = {"train": 0.70, "validation": 0.15, "test": 0.15}

rng = np.random.default_rng(SEED)
split_do_template = {}
for sinal, templates in sinal_template.groupby(sinal_template):
    ids = rng.permutation(templates.index.to_numpy())
    n_val = round(len(ids) * PROPORCAO["validation"])
    n_test = round(len(ids) * PROPORCAO["test"])
    for i, template_id in enumerate(ids):
        split_do_template[template_id] = "test" if i < n_test else "validation" if i < n_test + n_val else "train"

df["split"] = df["template_id"].map(split_do_template)

templates_por_split = df.groupby("split")["template_id"].unique()
assert not set(templates_por_split["train"]) & set(templates_por_split["test"]), "template no treino e no teste"
assert not set(templates_por_split["train"]) & set(templates_por_split["validation"]), "template no treino e na validação"

resumo = df.groupby("split").agg(exemplos=("alerta", "size"), templates=("template_id", "nunique"))
resumo["proporção"] = (resumo["exemplos"] / len(df)).map("{:.0%}".format)
resumo.reindex(PROPORCAO)

### 2.4.2 Conferir a distribuição nos splits

A proporção de cada severidade e de cada veredito deve ficar parecida nos três splits. Se o teste tiver muito menos críticos que o treino, por exemplo, o recall de "crítica" da Fase 6 fica pouco confiável.

In [ ]:
ordem = ["critica", "alta", "media", "baixa"]
print(pd.crosstab(df["split"], df["severidade"], normalize="index").reindex(index=list(PROPORCAO), columns=ordem).round(2))
print()
print(pd.crosstab(df["split"], df["veredito"], normalize="index").reindex(list(PROPORCAO)).round(2))

**Leitura:** as proporções ficam próximas nos três splits. O teste tem um pouco menos de críticos que o treino (8% contra 11%), mas ainda são cerca de 18 alertas críticos, de 3 templates de sinal forte diferentes. Isso é variação esperada quando se divide por template, e não por exemplo.

### 2.4.3 Publicar os splits no Hub

Montar o `DatasetDict` no mesmo formato do arquivo original (sem as colunas auxiliares da auditoria) e publicar como dataset **privado** na sua conta:

In [ ]:
from datasets import Dataset, DatasetDict, Features, Value

SPLITS_ID = f"{conta['name']}/alertas-soc-splits"

# Features explícitas mantêm a ordem das chaves da triagem (veredito, severidade, justificativa).
# Sem elas, o datasets ordena as chaves em ordem alfabética e o JSON de treino da Fase 3 mudaria de formato.
FEATURES = Features({
    "template_id": Value("string"),
    "alerta": Value("string"),
    "contexto": Value("string"),
    "triagem": {"veredito": Value("string"), "severidade": Value("string"), "justificativa": Value("string")},
})


def para_dataset(parte: pd.DataFrame) -> Dataset:
    registros = [
        {"template_id": r.template_id, "alerta": r.alerta, "contexto": r.contexto,
         "triagem": {"veredito": r.veredito, "severidade": r.severidade, "justificativa": r.justificativa}}
        for r in parte.itertuples()
    ]
    return Dataset.from_list(registros, features=FEATURES)


splits = DatasetDict({nome: para_dataset(df[df["split"] == nome]) for nome in PROPORCAO})
print(splits)

splits.push_to_hub(SPLITS_ID, private=True)

Recarregar do Hub para confirmar a publicação. As fases seguintes carregam os splits exatamente assim:

In [ ]:
splits = load_dataset(SPLITS_ID)
print(splits)
print(splits["test"][0])

## 2.5 Conclusão

Resultado da auditoria do histórico de triagens:

| Verificação | Encontrado | Ação |
|---|---|---|
| Grafias fora do padrão | 132 exemplos (categorias e espaços) | Padronizados |
| Triagem inválida | 24 exemplos | Removidos |
| Duplicatas de `alerta` + `contexto` | 58 (só 35 visíveis antes de padronizar) | Removidas |
| Rótulos inconsistentes com a política | 90, sendo 36 críticos rebaixados | Corrigidos pela política |
| Distribuição de classes | `critica` com 11% e `falso_positivo` com 8% | Mantida |

**Conclusão:** 1.478 exemplos limpos, divididos por template em treino (1.033), validação (224) e teste (221), com distribuição de severidade parecida nos três. O defeito mais grave era o de política: 6% dos rótulos contrariavam a política, e 40% deles eram alertas críticos rebaixados, exatamente o erro que a meta de recall de "crítica" quer evitar. Sem essa correção, o modelo aprenderia a repetir a inconsistência entre analistas que o fine-tuning deveria eliminar.